# 🚀 Модуль 13. Урок 2. Problem Framing: перевод размытых бизнес-вопросов в строгие дата-задачи

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_13_soft_skills_storytelling/lesson_02_problem_framing_for_ai.ipynb)

## 📌 Бизнес-контекст и методология
Заказчик почти никогда не приходит с четким ТЗ. Обычно он говорит: "Кажется, наши продажи падают, разберись".
Фреймворк **Issue Tree (Дерево проблем)** декомпозирует вопрос:
1. Выручка = Трафик $	imes$ Конверсия (CR) $	imes$ Средний чек (AOV).
2. Падение в каком именно сегменте? (Новые vs Повторные клиенты).
3. Проблема техническая (отказ платежей) или маркетинговая (нецелевой трафик)?

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
# Пример декомпозиции выручки по мультипликативной модели
def decompose_revenue(visitors, cr, aov):
    orders = visitors * cr
    revenue = orders * aov
    return {"visitors": visitors, "orders": int(orders), "revenue": revenue}

baseline = decompose_revenue(100_000, 0.03, 3000)
bad_month = decompose_revenue(100_000, 0.024, 3000) # CR просел с 3.0% до 2.4%

loss = baseline["revenue"] - bad_month["revenue"]
print(f"Базовая выручка:  {baseline['revenue']:,.0f} руб.")
print(f"Выручка в кризис: {bad_month['revenue']:,.0f} руб.")
print(f"Потери из-за просадки CR: {loss:,.0f} руб.")


Базовая выручка:  9,000,000 руб.
Выручка в кризис: 7,200,000 руб.
Потери из-за просадки CR: 1,800,000 руб.


### 🎯 Практическое задание 1: Оценка относительных потерь
Вычислите относительную долю потерянной выручки.

In [3]:
### ВАШ КОД ЗДЕСЬ
loss_pct = loss / baseline["revenue"]
print(f"Потеряно: {loss_pct:.1%} выручки")

assert round(loss_pct, 2) == 0.20, "Просадка CR на 20% дает 20% потерю выручки при неизменных прочих"
print("✅ Тест успешно пройден!")


Потеряно: 20.0% выручки
✅ Тест успешно пройден!


### 💼 Правило первой встречи
Никогда не начинайте писать SQL до тех пор, пока с заказчиком письменно не зафиксирован ожидаемый результат анализа.